# TetraFT — Kaggle runs

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`
- **B only:** Session A full `checkpoint-final`
- **P only:** Session B `checkpoint-final` (weights-only OK)
- **R5 / S / U:** code + FineWeb only — **fresh start, no ckpt**
- **L (layer map):** code + FineWeb + B `checkpoint-final`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run |

| SESSION | Preset / script | Resume | Gate |
|---------|-----------------|--------|------|
| **R5** | `scout_kl_r5_5m` LoRA only | **fresh** 1280 | ✅ **48.38** PASS |
| **L** | `run_layer_map.py` | B ckpt | role table |
| **U** | bundle R345 | fresh | ❌ FAIL — do not run |
| **A** / **B** | `heal_kl_50m` | see table | done ~34.38 |
| **P** | polish | B | ❌ FAIL |
| **S** | α/T scout | fresh | ⚪ null |

**R5 done:** end PPL **48.38** (gate 49.31).  
**Next:** fresh long KL + LoRA r=8 (preset TBD; gate &lt;34.38). Replay R5 with `SESSION="R5"`.  
Logic in `run_smoke.py` / `run_layer_map.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_50m" in SMOKE_PRESETS, "heal_kl_50m missing — refresh tetraft-code"
assert "scout_kl_5m" in SMOKE_PRESETS
assert "scout_kl_r5_5m" in SMOKE_PRESETS, "scout_kl_r5_5m missing — refresh tetraft-code"
assert (code_root / "run_layer_map.py").is_file(), "run_layer_map.py missing — refresh tetraft-code"
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
from run_layer_map import main as run_layer_map_main
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION: "R5" LoRA-only | "L" layer map | "U" bundle FAIL | A | B | P | S
# =============================================================================
SESSION = "R5"  # <-- R5 | L | U | A | B | P | S

# SESSION=S only (historical; static α/T null)
SCOUT_ALPHA = 0.3
SCOUT_TEMPERATURE = 2.0
SCOUT_TAG = "a03_t2"

# SESSION=L only
LAYER_MAP_CHECKPOINT = None  # e.g. "/kaggle/input/.../checkpoint-final"
LAYER_MAP_FP_MASK_TOPK = 8
LAYER_MAP_SKIP_PPL = False

CLEAR_OUTPUT = True

if SESSION.upper() == "R5":
    # LoRA only @ ~5.24M — no pre_rms, no weight_calib. Gate < 49.31
    PRESET = "scout_kl_r5_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None  # preset 8
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_scout_kl_r5_5m"
    print("R5-only: lora_rank=8, pre_rms=False, weight_calib=none")
    print("gate: end PPL < 49.31; abort if @λ=1 PPL ≫ 500")
elif SESSION.upper() == "U":
    raise RuntimeError(
        "SESSION=U bundle R345 FAILED (PPL 1000+). Use SESSION=R5 instead."
    )
elif SESSION.upper() == "L":
    ckpt = LAYER_MAP_CHECKPOINT or str(find_file("checkpoint-final"))
    out_lm = Path("/kaggle/working/layer_map_b")
    if CLEAR_OUTPUT and out_lm.exists():
        shutil.rmtree(out_lm)
        print("cleared", out_lm)
    out_lm.mkdir(parents=True, exist_ok=True)
    argv = [
        "--checkpoint", ckpt,
        "--preset", "heal_kl_50m",
        "--val-data", str(val_path),
        "--max-eval-batches", "20",
        "--fp-mask-topk", str(int(LAYER_MAP_FP_MASK_TOPK)),
        "--output-dir", str(out_lm),
    ]
    if LAYER_MAP_SKIP_PPL:
        argv.append("--skip-ppl")
    print("layer map checkpoint:", ckpt)
    print("argv:", argv)
    rc = run_layer_map_main(argv)
    print("layer_map exit", rc)
    summary = out_lm / "layer_map_summary.json"
    if summary.is_file():
        import json
        with summary.open() as f:
            s = json.load(f)
        print("suggestion:", s.get("suggestion"))
        print("ppl_student:", s.get("ppl_student"), "ppl_fp_mask:", s.get("ppl_fp_mask"))
    raise SystemExit(rc)
elif SESSION.upper() == "A":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif SESSION.upper() == "B":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
    print("resume from:", RESUME)
elif SESSION.upper() == "P":
    PRESET = "polish_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_polish_kl_5m"
    print("polish resume from:", RESUME)
elif SESSION.upper() == "S":
    PRESET = "scout_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = float(SCOUT_ALPHA)
    DISTILL_TEMPERATURE = float(SCOUT_TEMPERATURE)
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = f"/kaggle/working/checkpoints_scout_kl_{SCOUT_TAG}"
    print(f"α/T scout: α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} tag={SCOUT_TAG}")
    print("note: static α/T null — prefer SESSION=R5")
else:
    raise ValueError("SESSION must be 'R5', 'L', 'U', 'A', 'B', 'P', or 'S'")

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    schedule_max_steps=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=DISTILL_ALPHA,
    distill_temperature=DISTILL_TEMPERATURE,
    quant_reg_beta=None,
    pre_rms=PRE_RMS,
    no_pre_rms=NO_PRE_RMS,
    weight_calib=WEIGHT_CALIB,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} resume={RESUME} out={OUTPUT_DIR}"
)
print("note: KL loads frozen FP teacher (~2× VRAM)")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    inv = results["inventory_summary"]
    print("inventory", inv)
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear — GDN skip may be off")
ppl = results.get("ppl_after_smoke")
if ppl is not None:
    ref = results.get("ppl_original") or results.get("ppl_original_ref") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f} (ref orig {ref})")
    if SESSION.upper() == "A":
        print(f"Session A mid PPL={ppl:.2f} — go/no-go: continue B if ≲50–52 and falling")
    elif SESSION.upper() == "B":
        print(f"Session B final PPL={ppl:.2f} — bar CE heal_50m ~43.77; strong if ≲35")
    elif SESSION.upper() == "P":
        print(f"Polish final PPL={ppl:.2f} — gate < 34.38 (FAIL expected — stop polish)")
    elif SESSION.upper() == "R5":
        gate = 49.31
        print(f"R5 LoRA final PPL={ppl:.2f} — gate < {gate}")
        if ppl < gate:
            print("PASS — fresh long KL with lora_rank=8 (not polish B)")
        else:
            print("NO PASS — R5 null @ 5M; see RESULTS.md §5.9")
    else:
        gate = 49.31
        print(f"α/T scout final PPL={ppl:.2f} — gate < {gate} (historical)")
        if ppl < gate:
            print(f"PASS — lock α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE}")
        else:
            print("NO PASS — keep α=0.5 T=2; prefer SESSION=R5")

### R5-only scout (SESSION=R5) ✅ PASS 48.38

Preset `scout_kl_r5_5m`:
- **R5** `lora_rank=8` α=8 (B=0 init)
- **R3/R4 off**

Result: end PPL **48.38** &lt; 49.31. Best 5M scout. Promote: long KL + LoRA (not polish B).

Abort if @λ=1 (~step 256) PPL ≫ 500 (bundle-like collapse).

### Bundle (SESSION=U) — FAIL

Do not run. Notebook raises if `SESSION="U"`.

### Layer map (SESSION=L)

Attach B `checkpoint-final` + FineWeb. Artifacts under `/kaggle/working/layer_map_b/`.

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| scout_kl_5m | 5.2M | **~49.31** |
| CE heal_50m | 50M | ~43.77 |
| heal_kl_50m A+B | 50M | **~34.38** |
| polish / α/T / bundle R345 | — | FAIL / null / FAIL |

Record new PPL in `RESULTS.md`.
